In [2]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as patches
from matplotlib import gridspec
import matplotlib.colors as mcolors
import os
import pandas as pd
import seaborn as sns

mpl.use('Agg')
plt.rcParams['font.size'] = 16
plt.rcParams['text.usetex'] = True
plt.rcParams['font.family'] = "Computer Modern Roman"
plt.rcParams["text.latex.preamble"] = r'\usepackage{amsfonts} \usepackage{amssymb} \usepackage{amsmath}'

In [3]:
# Load the experiments
experiments = pd.read_csv('all_exp.csv')
folders_index = experiments['folder'].tolist()
labels_index = experiments['label'].tolist()
colors_index = experiments['color'].tolist()
triggers_path_index = experiments['triggers_path'].tolist()

In [7]:
# VGG
index = [354,429,420]
# ResNet
# index = [371,430,434,425]
labels = ["No WM","Vanilla","BlackCATT", "BlackCATT+FR"]
# index = [430,434,425]
# labels = ["Vanilla","BlackCATT", "BlackCATT+FR"]
# Number triggers
# index = [432,425,418]
# labels = [r"$T=100$", r"$T=250$", r"$T=500$"]
# Number DOs
# index = [425,428,431]
# labels = [r"$N=20$", r"$N=40$", r"$N=60$"]
# Optimization rounds
# index = [425,426,424]
# labels = ["1 round","2 rounds","5 rounds"]
# Effect of aux dataset
index = [437,439,438]
labels = ["WikiArt", "CIFAR-100", "TinyImageNet"]
# index = [441,440]
# labels = ["No WM", "BlackCATT"]
# Ablation
# index = [425,434,436,435]
# labels = ["BlackCATT+FR", "BlackCATT", r"BlackCATT+FR w/o $\nabla_{\mathbf{x}^{(r)}}$", r"BlackCATT+FR w/o $L_\text{CA}$"]
# index = [425,436,434,435]
# labels = ["BlackCATT+FR", r"BlackCATT+FR w/o $\nabla_{\mathbf{x}^{(r)}}$", "BlackCATT", r"BlackCATT+FR w/o $L_\text{CA}$"]
# Benign triggers
# index = [371,425,433]
# labels = ["No WM", "BlackCATT+FR", "BlackCATT(S)+FR"]


folders = [folders_index[i] for i in index]
# labels = [labels_index[i] for i in index]
colors = [colors_index[i] for i in index]
triggers_path = [triggers_path_index[i] for i in index]

#### Evol

In [6]:
df_all = pd.DataFrame({"strategy":[],"round":[],"loss":[],"accuracy":[],"trigger":[],"mav":[],"fn":[]})
fig, axes = plt.subplots(1,3, figsize=(7,2))

# Read the metrics file
for folder in folders:
    folder_aux = "/run/user/1000/gvfs/sftp:host=172.19.57.230,user=erodriguez/Neuronal/FL_models/new_version/" + folder.split("/")[-2] + "/" 
    for i_cid in range(10):
        # try:
        with open(folder_aux + "metrics_"+str(i_cid)+".csv", "r") as file:
            data = file.readlines()
            for round, line in enumerate(data):
                loss_i = float(line.split(",")[0])
                accuracy_i = float(line.split(",")[1])
                accuracy_t_i =  float(line.split(",")[2])
                mav_i =  float(line.split(",")[3])
                try:
                    fn_i =  float(line.split(",")[4] =="True")
                except:
                    # print("No fn info")
                    fn_i = 1
                df = pd.DataFrame({"strategy":[labels[folders.index(folder)]],
                    "round":[(round+1)*25],
                    "loss":[loss_i],
                    "accuracy":[accuracy_i],
                    "trigger":[accuracy_t_i],
                    "mav":[mav_i],
                    "fn":[fn_i]})
                df_all = pd.concat([df_all,df])
        # except:
        #     pass
        
thr_mav = []
# Calculate threshold MAV for each strategy
for i, folder in enumerate(folders):
    strategy_data = df_all[df_all["strategy"] == labels[i]]
    
    # Group by round and calculate mean MAV for each round
    avg_fn_by_round = strategy_data.groupby("round")["fn"].mean()
    
    # Find first round where average MAV goes below 0.5 FNR
    threshold_round = None
    for round_num, avg_fn in avg_fn_by_round.items():
        if avg_fn < 0.5:
            threshold_round = round_num
            break
    
    thr_mav.append(threshold_round)

sns.lineplot(data=df_all,x="round",y="accuracy",hue="strategy",palette=colors,ax=axes[0])

axes[0].set_xlabel("Round")
axes[0].set_ylabel("Accuracy")
# axes[0].set_yticks(np.arange(0.3, 0.71, 0.1))
axes[0].set_yticks(np.arange(0.5, 0.91, 0.1))
axes[0].set_xticks(np.arange(0, 1501, 500))
axes[0].set_xticklabels(np.arange(0, 1501, 500),rotation=45)
axes[0].set_xlim([0,1500])
# axes[0].set_ylim([0.35,0.76])
axes[0].set_ylim([0.45,0.92])
axes[0].grid(alpha=0.5)
axes[0].legend().remove()

sns.lineplot(data=df_all,x="round",y="trigger",hue="strategy",palette=colors,ax=axes[1])
axes[1].set_xlabel("Round")
axes[1].set_ylabel("Trigger Accuracy")
axes[1].set_yticks(np.arange(0, 1.01, 0.2))
axes[1].set_xticks(np.arange(0, 1501, 500))
axes[1].set_xticklabels(np.arange(0, 1501, 500),rotation=45)
axes[1].set_xlim([0,1500])
axes[1].grid(alpha=0.5)
axes[1].legend().remove()

sns.lineplot(data=df_all,x="round",y="mav",hue="strategy",palette=colors,ax=axes[2])
axes[2].set_xlabel("Round")
axes[2].set_ylabel(r"MAV in $\mathcal{C}^2$")
axes[2].set_yticks(np.arange(0, 1.01, 0.2))
axes[2].set_xticks(np.arange(0, 1501, 500))
axes[2].set_xticklabels(np.arange(0, 1501, 500),rotation=45)
axes[2].set_xlim([0,1500])
axes[2].grid(alpha=0.5)
axes[2].legend().remove()

for i in range(len(folders)):
    if thr_mav[i] is not None:  # Only draw line if threshold was found
        axes[0].vlines(x=thr_mav[i],ymin=0,ymax=1,colors=colors[i],alpha=0.5,linestyles='dashed',label="Threshold " + labels[i])
        axes[1].vlines(x=thr_mav[i],ymin=0,ymax=1,colors=colors[i],alpha=0.5,linestyles='dashed',label="Threshold " + labels[i])
        axes[2].vlines(x=thr_mav[i],ymin=0,ymax=1,colors=colors[i],alpha=0.5,linestyles='dashed',label="Threshold " + labels[i])

# legend = axes[1].legend( loc='upper center', bbox_to_anchor=(0.5, 3),
#           ncol=3, fancybox=True)

plt.subplots_adjust(wspace=0.6)
# plt.tight_layout()
plt.show()


# def export_legend(legend, filename="legend.png", expand=[-5,-5,5,5]):
#     fig  = legend.figure
#     fig.canvas.draw()
#     bbox  = legend.get_window_extent()
#     bbox = bbox.from_extents(*(bbox.extents + np.array(expand)))
#     bbox = bbox.transformed(fig.dpi_scale_trans.inverted())
#     fig.savefig("./GRAPHS/legend_N.pdf", dpi="figure", bbox_inches=bbox, format='pdf')

# export_legend(legend)

# plt.savefig("./GRAPHS/evol_resnet183x3_cifar100.pdf", bbox_inches='tight', format='pdf')
plt.savefig("./GRAPHS/evol_vgg16_cifar10.pdf", bbox_inches='tight', format='pdf')
# plt.savefig("./GRAPHS/evol_N.pdf", bbox_inches='tight', format='pdf')
# plt.savefig("./GRAPHS/evol_ablation.pdf", bbox_inches='tight', format='pdf')
# plt.savefig("./GRAPHS/evol_b9.pdf", bbox_inches='tight', format='pdf')

/tmp/ipykernel_38999/3356787129.py:94: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


#### Only Acc Evol

In [8]:
df_all = pd.DataFrame({"strategy":[],"round":[],"loss":[],"accuracy":[],"trigger":[],"mav":[]})
fig, axes = plt.subplots(1,1, figsize=(7,2))

# Read the metrics file
for folder in folders:
    folder_aux = "/run/user/1000/gvfs/sftp:host=172.19.57.230,user=erodriguez/Neuronal/FL_models/new_version/" + folder.split("/")[-2] + "/" 
    for i_cid in range(10):
        # try:
        with open(folder_aux + "metrics_"+str(i_cid)+".csv", "r") as file:
            data = file.readlines()
            for round, line in enumerate(data):
                loss_i = float(line.split(",")[0])
                accuracy_i = float(line.split(",")[1])
                accuracy_t_i =  float(line.split(",")[2])
                mav_i =  float(line.split(",")[3])
                try:
                    fn_i =  float(line.split(",")[4] =="True")
                except:
                    # print("No fn info")
                    fn_i = 1
                df = pd.DataFrame({"strategy":[labels[folders.index(folder)]],
                    "round":[(round+1)*25],
                    "loss":[loss_i],
                    "accuracy":[accuracy_i],
                    "trigger":[accuracy_t_i],
                    "mav":[mav_i],
                    "fn":[fn_i]})
                df_all = pd.concat([df_all,df])
        # except:
        #     pass
        
thr_mav = []
# Calculate threshold MAV for each strategy
for i, folder in enumerate(folders):
    strategy_data = df_all[df_all["strategy"] == labels[i]]
    
    # Group by round and calculate mean MAV for each round
    avg_fn_by_round = strategy_data.groupby("round")["fn"].mean()
    
    # Find first round where average MAV goes below 0.5 FNR
    threshold_round = None
    for round_num, avg_fn in avg_fn_by_round.items():
        if avg_fn < 0.5:
            threshold_round = round_num
            break
    
    thr_mav.append(threshold_round)

sns.lineplot(data=df_all,x="round",y="accuracy",hue="strategy",palette=colors,ax=axes)

axes.set_xlabel("Round")
axes.set_ylabel("Accuracy")
axes.set_yticks(np.arange(0.3, 0.71, 0.1))
# axes[0].set_yticks(np.arange(0.5, 0.91, 0.1))
# axes.set_yticks(np.arange(0.8, 0.951, 0.04))
axes.set_xticks(np.arange(0, 1501, 250))
axes.set_xlim([0,1500])
axes.set_ylim([0.35,0.76])
# axes[0].set_ylim([0.45,0.92])
# axes.set_ylim([0.79,0.955])
axes.grid(alpha=0.5)
axes.legend().remove()

# for i in range(len(folders)):
#     if thr_mav[i] is not None:  # Only draw line if threshold was found
#         axes.vlines(x=thr_mav[i],ymin=0,ymax=1,colors=colors[i],alpha=0.5,linestyles='dashed',label="Threshold " + labels[i])

legend = axes.legend( loc='upper center', bbox_to_anchor=(0.5, 3),
          ncol=3, fancybox=True)

plt.subplots_adjust(wspace=0.6)
# plt.tight_layout()
plt.show()


def export_legend(legend, filename="legend.png", expand=[-5,-5,5,5]):
    fig  = legend.figure
    fig.canvas.draw()
    bbox  = legend.get_window_extent()
    bbox = bbox.from_extents(*(bbox.extents + np.array(expand)))
    bbox = bbox.transformed(fig.dpi_scale_trans.inverted())
    fig.savefig("./GRAPHS/legend_datasets.pdf", dpi="figure", bbox_inches=bbox, format='pdf')

export_legend(legend)

# plt.savefig("./GRAPHS/evol_aux_datasets.pdf", bbox_inches='tight', format='pdf')

/tmp/ipykernel_38999/816440773.py:73: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


#### Accusation metrics

In [25]:
df_all = pd.DataFrame({"strategy":[],"c":[],"m_needed":[],"fn":[]})
df_all_r = pd.DataFrame({"strategy":[],"c":[],"m_needed":[],"fn":[]})

fig, axes = plt.subplots(1, 2, figsize=(7,2.5))

max_colluders = 5

for folder in folders[1:]:
    print("Loading folder: ",folder)
    df = pd.read_csv("df_all_average_"+str(index[folders.index(folder)])+".csv")  
    df_random = pd.read_csv("df_all_randomselect_"+str(index[folders.index(folder)])+".csv")  

    df_all = pd.concat([df_all,df])
    df_all_r = pd.concat([df_all_r,df_random])


df_all = df_all[df_all["c"]<=max_colluders]
df_all_r = df_all_r[df_all_r["c"]<=max_colluders]

sns.lineplot(data=df_all,x="c",y="fn",hue="strategy",errorbar=None,palette=colors[1:],ax=axes[1])
sns.lineplot(data=df_all,x="c",y="m_needed",hue="strategy",palette=colors[1:],ax=axes[0])
sns.lineplot(data=df_all_r,x="c",y="fn",hue="strategy",errorbar=None,linestyle='dashed',palette=colors[1:],ax=axes[1])
sns.lineplot(data=df_all_r,x="c",y="m_needed",hue="strategy",linestyle='dashed',palette=colors[1:],ax=axes[0])
axes[0].set_xlabel(r"c (\# of colluders)")
axes[0].set_ylabel(r"$t^*$")
axes[0].set_yticks(np.arange(0,251,50))
axes[0].set_xticks(np.arange(1,max_colluders+1))
axes[0].grid(alpha=0.5)
axes[1].set_xlabel(r"c (\# of colluders)")
axes[1].set_ylabel("FNR")
axes[1].set_ylim(-0.01,1.01)
axes[1].set_yticks(np.arange(0,1.1,0.2))
axes[1].set_xticks(np.arange(1,max_colluders+1))
axes[1].grid(alpha=0.5)
fig.align_ylabels(axes[:])
axes[0].legend().remove()
axes[1].legend().remove()

# legend = axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, 1.75),
#           ncol=3, fancybox=True)

plt.tight_layout()
plt.show()

# def export_legend(legend, filename="legend.png", expand=[-5,-5,5,5]):
#     fig  = legend.figure
#     fig.canvas.draw()
#     bbox  = legend.get_window_extent()
#     bbox = bbox.from_extents(*(bbox.extents + np.array(expand)))
#     bbox = bbox.transformed(fig.dpi_scale_trans.inverted())
#     fig.savefig(filename, dpi="figure", bbox_inches=bbox)

# export_legend(legend)
# plt.show()
plt.savefig("./GRAPHS/m_vgg16_cifar10.pdf", bbox_inches='tight', format='pdf')
# plt.savefig("./GRAPHS/m_resnet183x3_cifar100.pdf", bbox_inches='tight', format='pdf')


Loading folder:  ../../FL_models/new_version/VGG16_CIFAR10_20users_0.5k_250m_64mbs_0.01mlr_0.0001tlr_39mbatches_0lambdaregCOL0_0talphaTRIG_wikiartCLAvgKL0_0troundsBest/
Loading folder:  ../../FL_models/new_version/VGG16_CIFAR10_20users_0.5k_250m_64mbs_0.01mlr_0.0001tlr_39mbatches_0.1lambdaregCOL5_64talphaTRIG_wikiartCLAvgKL0_1troundsBest/


/tmp/ipykernel_13431/1488042376.py:43: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


#### All in one

In [4]:
df_all = pd.DataFrame({"strategy":[],"round":[],"loss":[],"accuracy":[],"trigger":[],"mav":[]})
fig, axes = plt.subplots(1,2, figsize=(7,2))

# Read the metrics file
for folder in folders:
    folder_aux = "/run/user/1000/gvfs/sftp:host=172.19.57.230,user=erodriguez/Neuronal/FL_models/new_version/" + folder.split("/")[-2] + "/" 
    for i_cid in range(10):
        # try:
        with open(folder_aux + "metrics_"+str(i_cid)+".csv", "r") as file:
            data = file.readlines()
            for round, line in enumerate(data):
                loss_i = float(line.split(",")[0])
                accuracy_i = float(line.split(",")[1])
                accuracy_t_i =  float(line.split(",")[2])
                mav_i =  float(line.split(",")[3])
                try:
                    fn_i =  float(line.split(",")[4] =="True")
                except:
                    # print("No fn info")
                    fn_i = 0
                df = pd.DataFrame({"strategy":[labels[folders.index(folder)]],
                    "round":[(round+1)*25],
                    "loss":[loss_i],
                    "accuracy":[accuracy_i],
                    "trigger":[accuracy_t_i],
                    "mav":[mav_i],
                    "fn":[fn_i]})
                df_all = pd.concat([df_all,df])
        # except:
        #     pass
        
thr_mav = []
# Calculate threshold MAV for each strategy
for i, folder in enumerate(folders):
    strategy_data = df_all[df_all["strategy"] == labels[i]]
    
    # Group by round and calculate mean MAV for each round
    avg_fn_by_round = strategy_data.groupby("round")["fn"].mean()
    
    # Find first round where average MAV goes below 0.5 FNR
    threshold_round = None
    for round_num, avg_fn in avg_fn_by_round.items():
        if avg_fn < 0.5:
            threshold_round = round_num
            break
    
    thr_mav.append(threshold_round)

sns.lineplot(data=df_all,x="round",y="accuracy",hue="strategy",palette=colors,ax=axes[0])

axes[0].set_xlabel("Round")
axes[0].set_ylabel("Accuracy")
# axes[0].set_yticks(np.arange(0.2, 0.51, 0.05))
axes[0].set_yticks(np.arange(0.3, 0.71, 0.1))
# axes[0].set_yticks(np.arange(0.5, 0.91, 0.1))
# axes[0].set_yticks(np.arange(0.85, 0.946, 0.02))
# axes[0,0].set_yticks(np.arange(0.65, 0.851, 0.05))
axes[0].set_xticks(np.arange(0, 1501, 500))
axes[0].set_xlim([0,1500])
# axes[0].set_ylim([0.2,0.52])
axes[0].set_ylim([0.35,0.76])
# axes[0].set_ylim([0.25,0.62])
# axes[0].set_ylim([0.45,0.92])
# axes[0].set_ylim([0.85,0.95])
axes[0].grid(alpha=0.5)
axes[0].legend().remove()

for i in range(len(folders)):
    if thr_mav[i] is not None:  # Only draw line if threshold was found
        axes[0].vlines(x=thr_mav[i],ymin=0,ymax=1,colors=colors[i],alpha=0.5,linestyles='dashed',label="Threshold " + labels[i])

df_all = pd.DataFrame({"strategy":[],"c":[],"m_needed":[],"m_needed_ratio":[],"fn":[]})

max_colluders = 5

for folder in folders:
    print("Loading folder: ",folder)
    df = pd.read_csv("df_all_average_"+str(index[folders.index(folder)])+".csv")  
    m = folder.split("_0.5k_")[-1].split("m")[0]
    df["m_needed_ratio"] = df["m_needed"] / int(m)
    df_all = pd.concat([df_all,df])

df_all = df_all[df_all["c"]<=max_colluders]

sns.lineplot(data=df_all,x="c",y="m_needed_ratio",hue="strategy",palette=colors,ax=axes[1])

axes[1].set_xlabel(r"c (\# of colluders)")
axes[1].set_ylabel(r"$t^* / T$")
axes[1].set_ylim(-0.01,1.01)
axes[1].set_yticks(np.arange(0,1.1,0.2))
axes[1].set_xticks(np.arange(1,max_colluders+1))
axes[1].grid(alpha=0.5)
axes[1].legend().remove()

# legend = axes[0].legend( loc='upper center', bbox_to_anchor=(0.5, 3),
#           ncol=3, fancybox=True)

plt.tight_layout()
plt.show()

# def export_legend(legend, filename="legend.png", expand=[-5,-5,5,5]):
#     fig  = legend.figure
#     fig.canvas.draw()
#     bbox  = legend.get_window_extent()
#     bbox = bbox.from_extents(*(bbox.extents + np.array(expand)))
#     bbox = bbox.transformed(fig.dpi_scale_trans.inverted())
#     fig.savefig("./GRAPHS/legend_M.pdf", dpi="figure", bbox_inches=bbox, format='pdf')

# export_legend(legend)
plt.savefig("./GRAPHS/all_M.pdf", bbox_inches='tight', format='pdf')


Loading folder:  ../../FL_models/new_version/ResNet183x3_CIFAR100_20users_0.5k_100m_64mbs_0.01mlr_0.0001tlr_39mbatches_0.1lambdaregCOL5_64talphaTRIG_wikiartCLAvgKL0.1_1troundsBest/
Loading folder:  ../../FL_models/new_version/ResNet183x3_CIFAR100_20users_0.5k_250m_64mbs_0.01mlr_0.0001tlr_39mbatches_0.1lambdaregCOL5_64talphaTRIG_wikiartCLAvgKL0.1_1troundsBest/
Loading folder:  ../../FL_models/new_version/ResNet183x3_CIFAR100_20users_0.5k_500m_64mbs_0.01mlr_0.0001tlr_39mbatches_0.1lambdaregCOL5_64talphaTRIG_wikiartCLAvgKL0.1_1troundsBest/


/tmp/ipykernel_38999/2281360803.py:99: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


#### Fine-pruning


In [27]:
index = [425]

df_all = pd.read_csv("df_all_fine_prun_"+str(index[0])+"_new.csv")  
df_all = df_all[df_all["c"]<=5]

fig, axes = plt.subplots(1, 1, figsize=(7,3))

folders = [folders_index[i] for i in index]
labels = [labels_index[i] for i in index]
colors = [colors_index[i] for i in index]

sns.lineplot(data=df_all,x="pruning_rate",y="fn",hue="c",palette="flare_r",errorbar=None,ax=axes)
ax2 = plt.twinx()
sns.lineplot(data=df_all,x="pruning_rate",y="accuracy",hue="c",palette="flare_r",errorbar=None,ax=ax2, linestyle='--')

axes.set_xticks(sorted(df_all['pruning_rate'].unique()))
axes.set_xticklabels(sorted(df_all['pruning_rate'].unique()), rotation=45)
ax2.set_ylabel("Task Accuracy")
axes.set_ylabel("FNR")
axes.set_yticks(np.arange(0,1.1,0.2))
axes.set_xlabel("Pruning rate")
ax2.set_xlabel("Pruning rate")
ax2.legend().remove()
axes.legend().remove()
axes.grid(alpha=0.5)

legend_text = [r"$\mathcal{C}^"+str(i)+r"$" for i in range(1,6)]
legend = axes.legend(legend_text,loc='upper center', bbox_to_anchor=(0.5, 3),
          ncol=5, fancybox=True)

plt.tight_layout()
plt.show()

def export_legend(legend, filename="legend.png", expand=[-5,-5,5,5]):
    fig  = legend.figure
    fig.canvas.draw()
    bbox  = legend.get_window_extent()
    bbox = bbox.from_extents(*(bbox.extents + np.array(expand)))
    bbox = bbox.transformed(fig.dpi_scale_trans.inverted())
    fig.savefig("./GRAPHS/legend_fine.pdf", dpi="figure", bbox_inches=bbox, format='pdf')

export_legend(legend)
plt.show()

# plt.savefig("./GRAPHS/m_fine_prun_vgg16_cifar10.pdf", bbox_inches='tight', format='pdf')
# plt.savefig("./GRAPHS/m_fine_prun_resnet183x3_cifar100.pdf", bbox_inches='tight', format='pdf')


/tmp/ipykernel_13431/2891969566.py:31: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all axes decorations.
  plt.tight_layout()
/tmp/ipykernel_13431/2891969566.py:32: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/tmp/ipykernel_13431/2891969566.py:43: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


#### Trigger evolution

In [ ]:
folder = folders_index[425]
folder_aux = "/run/user/1000/gvfs/sftp:host=172.19.57.230,user=erodriguez/Neuronal/FL_models/new_version/" + folder.split("/")[-2] + "/" 

t_i = 23
plt.figure(figsize=(8,2))
plt.subplot(1,4,1)
plt.ylabel("Trigger "+str(t_i))
for i in range(0,1501,500):
    # try:
        trigger_i = np.load(folder_aux + "trigger_round_"+str(i)+".npy").astype(np.uint8)
        plt.subplot(1,4,i//500+1)
        plt.imshow(trigger_i[t_i])
        plt.title("Round " +str(i),fontsize=14)
        plt.axis("off")
    # except:
    #     continue
plt.savefig("./GRAPHS/example_trigger.pdf", bbox_inches='tight', format='pdf')


#### Unique triggers

In [23]:
index = 381
folder = folders_index[index]

df_all = pd.read_csv("df_all_unique_"+str(index)+".csv")  
fig, axes = plt.subplots(1, 1, figsize=(7,3))

print("Loading folder: ",folder)

strategies = ["No Attacks", "Fine-tuning", "Pruning and\nFine-tuning", "Averaging\nTwo Models"]

sns.barplot(data=df_all, x="strategy", y="accuracy_t", ax=axes)
ax2 = plt.twinx()
sns.lineplot(data=df_all, x="strategy", y="acc", ax=ax2, color='black', marker='o', label='CIFAR10 Accuracy')
axes.hlines(y=0.1, xmin=-0.5, xmax=3.5, color='red', alpha=0.5, linestyle='dashed', label='Random Guess')
plt.xlabel("")
axes.set_ylabel("Trigger Accuracy")
ax2.set_ylabel("Task Accuracy")
# ax2.set_ylim([0,1])
ax2.legend().remove()
axes.legend().remove()
plt.grid(alpha=0.5)

axes.set_yticks(np.arange(0, 1.1, 0.2))
ax2.set_yticks(np.arange(0.8, 0.91, 0.02))
axes.set_xlabel("")
ax2.set_xlabel("")
axes.set_xlim([-0.5,3.5])
ax2.set_xlim([-0.5,3.5])

# legend = axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, 1.75),
#           ncol=2, fancybox=True)

plt.tight_layout()
plt.show()


# def export_legend(legend, filename="legend.png", expand=[-5,-5,5,5]):
#     fig  = legend.figure
#     fig.canvas.draw()
#     bbox  = legend.get_window_extent()
#     bbox = bbox.from_extents(*(bbox.extents + np.array(expand)))
#     bbox = bbox.transformed(fig.dpi_scale_trans.inverted())
#     fig.savefig(filename, dpi="figure", bbox_inches=bbox)

# export_legend(legend)
# plt.show()

plt.savefig("./GRAPHS/collusion_comparison.pdf", bbox_inches='tight', format='pdf')


/tmp/ipykernel_99071/598426918.py:5: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, axes = plt.subplots(1, 1, figsize=(7,3))


Loading folder:  ../../FL_models/new_version/TESTEpNIDA_STATE_VGG16_CIFAR10_20users_100Um_64mbs_0.01mlr_0.0001tlr_39mbatches/


/tmp/ipykernel_99071/598426918.py:34: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
